In [ ]:
import os
import sys
from google.colab import drive, userdata

# 1. Mount Cloud Storage Base
print("🔄 Connecting to Google Drive Cluster...")
drive.mount('/content/drive', force_remount=True)

# 2. Extract Secure Access Token from Secrets Manager
print("🔑 Authenticating Git Credentials...")
try:
    token = userdata.get("Github_Token")
except Exception:
    raise ValueError("❌ Please add 'Github_Token' to your Colab Left Sidebar Secrets (🔑 icon) before running.")

# 3. Synchronize Repository
REPO_DIR = "/content/economic-news-sentiment"
if not os.path.exists(REPO_DIR):
    print("📥 Clowning repository down to working machine runtime...")
    !git clone https://{token}@github.com/Nas-Mohd/economic-news-sentiment.git
else:
    print("🔄 Repo folder already exists. Checking current status...")

# 4. Bind Global Git Profiling Rules
!git config --global user.email "anasamohammad.school@gmail.com"
!git config --global user.name "Nas-Mohd"

# 5. Path Python Environment Directly to Repo Tree
sys.path.append(REPO_DIR)

# 6. Sanity Status Check
print("\n🛰️ Current Repository Version Track Tracking Status:")
!git -C {REPO_DIR} status

# 7. Environment Dependency Assurance
!pip install -q transformers[torch] datasets accelerate scikit-learn pandas numpy tqdm

🔄 Connecting to Google Drive Cluster...
Mounted at /content/drive
🔑 Authenticating Git Credentials...
📥 Clowning repository down to working machine runtime...
Cloning into 'economic-news-sentiment'...
remote: Enumerating objects: 616, done.
remote: Counting objects: 100% (233/233), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 616 (delta 141), reused 138 (delta 65), pack-reused 383 (from 1)
Receiving objects: 100% (616/616), 8.75 MiB | 7.97 MiB/s, done.
Resolving deltas: 100% (384/384), done.

🛰️ Current Repository Version Track Tracking Status:
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [ ]:
!git -C {REPO_DIR} pull

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 9 (delta 5), reused 9 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 19.27 KiB | 896.00 KiB/s, done.
From https://github.com/Nas-Mohd/economic-news-sentiment
   4601b20..f2786be  main       -> origin/main
Updating 4601b20..f2786be
Fast-forward
 .../data/finbert_absa_synthetic_training_ready.csv | 351 +++++++++++++++++++++
 webscraping/generate_synthetic_mixed.py            | 154 +++++++++
 webscraping/synthetic_generation.log               |  17 +
 webscraping/synthetic_mixed_sentences_fast.csv     | 351 +++++++++++++++++++++
 4 files changed, 873 insertions(+)
 create mode 100644 training/data/finbert_absa_synthetic_training_ready.csv
 create mode 100644 webscraping/generate_synthetic_mixed.py
 create mode 100644 webscraping/synthetic_generation.log
 create mode 100644 webscraping/synthetic_mixed_sentences_fast.csv


In [ ]:
import os
import re
import pandas as pd
import numpy as np

# Define your data pipeline paths
FILE_PATHS = {
    "Original Core Data": "/content/economic-news-sentiment/training/data/finbert_absa_training_ready.csv",
    "Checkpoint Patch 2": "/content/economic-news-sentiment/training/data/finbert_absa_training_ready_2.csv",
    "Synthetic Mixed Data": "/content/economic-news-sentiment/training/data/finbert_absa_synthetic_training_ready.csv"
}

DOMAIN_LABELS = [
    "Monetary_Financial", "Inflation_Prices", "Real_Economic_Activity",
    "Labor_Consumption", "Fiscal_Government", "External_Sector"
]

def analyze_file_profile(file_path):
    if not os.path.exists(file_path):
        return None

    df = pd.read_csv(file_path)
    total_rows = len(df)

    # 1. Structural Class Classifications (Using 1=Pos, 2=Neg, 3=Neu, 0=Abstain mapping)
    silent_count = 0
    single_count = 0
    multi_uniform_count = 0
    mixed_count = 0

    for _, row in df.iterrows():
        # Handle conversion safely whether columns contain floats (1.0) or ints (1)
        active_sentiments = [int(row[a]) for a in DOMAIN_LABELS if int(row[a]) in [1, 2, 3]]

        if len(active_sentiments) == 0:
            silent_count += 1
        elif len(active_sentiments) == 1:
            single_count += 1
        elif len(set(active_sentiments)) > 1:
            mixed_count += 1
        else:
            multi_uniform_count += 1

    # 2. Extract Active Volume Per Macro Aspect Domain
    aspect_distribution = {}
    for aspect in DOMAIN_LABELS:
        # Count rows where aspect is NOT 0 (Abstain)
        active_total = df[aspect].isin([1, 2, 3]).sum()
        clean_name = re.sub(r'_+', ' ', aspect).strip()
        aspect_distribution[clean_name] = active_total

    return {
        "total_rows": total_rows,
        "profiles": {
            "Silent (Abstain All)": silent_count,
            "Single-Aspect Focus": single_count,
            "Multi-Aspect (Uniform)": multi_uniform_count,
            "True Mixed Sentiment": mixed_count
        },
        "aspects": aspect_distribution
    }

print("==================== 🩺 MULTI-FILE ABSA DATA AUDIT ====================")

compiled_metrics = {}
for name, path in FILE_PATHS.items():
    metrics = analyze_file_profile(path)
    if metrics:
        compiled_metrics[name] = metrics
        print(f"✅ Loaded {name:<22} | Total Rows: {metrics['total_rows']}")
    else:
        print(f"⚠️ Missing {name:<22} at target path location: {path}")

print("\n================== ⚖️ SENTIMENT MATRIX COMPLEXITY COMPARISON ==================")
# Format columns dynamically
print(f"{'Data Profile Category':<25} | " + " | ".join([f"{name[:14]:<14}" for name in compiled_metrics.keys()]))
print("—" * 75)

profile_keys = ["Silent (Abstain All)", "Single-Aspect Focus", "Multi-Aspect (Uniform)", "True Mixed Sentiment"]
for key in profile_keys:
    row_str = f"{key:<25} | "
    shares = []
    for name, data in compiled_metrics.items():
        count = data["profiles"][key]
        pct = (count / data["total_rows"]) * 100
        shares.append(f"{count:<5} ({pct:4.1f}%)")
    row_str += " | ".join(shares)
    print(row_str)

print("\n==================== 📈 ACTIVE VOLUMES PER MACRO ASPECT ====================")
print(f"{'Target Macro Aspect':<25} | " + " | ".join([f"{name[:14]:<14}" for name in compiled_metrics.keys()]))
print("—" * 75)

clean_aspect_names = [re.sub(r'_+', ' ', a).strip() for a in DOMAIN_LABELS]
for aspect in clean_aspect_names:
    row_str = f"{aspect:<25} | "
    volumes = []
    for name, data in compiled_metrics.items():
        volume = data["aspects"][aspect]
        pct = (volume / data["total_rows"]) * 100
        volumes.append(f"{volume:<5} ({pct:4.1f}%)")
    row_str += " | ".join(volumes)
    print(row_str)

print("============================================================================")

==================== 🩺 MULTI-FILE ABSA DATA AUDIT ====================
✅ Loaded Original Core Data     | Total Rows: 3224
✅ Loaded Checkpoint Patch 2     | Total Rows: 3359
✅ Loaded Synthetic Mixed Data   | Total Rows: 350

================== ⚖️ SENTIMENT MATRIX COMPLEXITY COMPARISON ==================
Data Profile Category     | Original Core  | Checkpoint Pat | Synthetic Mixe
———————————————————————————————————————————————————————————————————————————
Silent (Abstain All)      | 697   (21.6%) | 1592  (47.4%) | 0     ( 0.0%)
Single-Aspect Focus       | 1528  (47.4%) | 837   (24.9%) | 8     ( 2.3%)
Multi-Aspect (Uniform)    | 707   (21.9%) | 813   (24.2%) | 0     ( 0.0%)
True Mixed Sentiment      | 292   ( 9.1%) | 117   ( 3.5%) | 342   (97.7%)

==================== 📈 ACTIVE VOLUMES PER MACRO ASPECT ====================
Target Macro Aspect       | Original Core  | Checkpoint Pat | Synthetic Mixe
———————————————————————————————————————————————————————————————————————————
Monetary Financia

In [ ]:
import pandas as pd

# 1. Load the three files
df_core = pd.read_csv("/content/economic-news-sentiment/training/data/finbert_absa_training_ready.csv")
df_patch = pd.read_csv("/content/economic-news-sentiment/training/data/finbert_absa_training_ready_2.csv")
df_synth = pd.read_csv("/content/economic-news-sentiment/training/data/finbert_absa_synthetic_training_ready.csv")

# Ensure all frames have identical columns
target_cols = ["Monetary_Financial", "Inflation_Prices", "Real_Economic_Activity",
               "Labor_Consumption", "Fiscal_Government", "External_Sector"]
all_cols = ["text"] + target_cols

df_raw_combined = pd.concat([df_core[all_cols], df_patch[all_cols], df_synth[all_cols]], ignore_index=True)

# 2. Segment rows by behavior
def get_profile(row):
    active = [int(row[a]) for a in target_cols if int(row[a]) in [1, 2, 3]]
    if len(active) == 0: return "silent"
    elif len(active) == 1: return "single"
    elif len(set(active)) > 1: return "mixed"
    else: return "uniform"

df_raw_combined['profile'] = df_raw_combined.apply(get_profile, axis=1)

df_silent = df_raw_combined[df_raw_combined['profile'] == "silent"]
df_single = df_raw_combined[df_raw_combined['profile'] == "single"]
df_uniform = df_raw_combined[df_raw_combined['profile'] == "uniform"]
df_mixed = df_raw_combined[df_raw_combined['profile'] == "mixed"]

# 3. Apply Heuristic Caps & Oversampling
# Cap silent rows at 800 (slashing down from the 2,289 raw pool)
df_silent_sampled = df_silent.sample(n=800, random_state=42)

# Oversample the true mixed rows (292 + 117 + 342 = 751 unique rows) 3x
df_mixed_oversampled = pd.concat([df_mixed] * 3, ignore_index=True)

# 4. Merge into final pre-split footprint
df_master = pd.concat([
    df_silent_sampled,
    df_single,
    df_uniform,
    df_mixed_oversampled
], ignore_index=True)

df_master.drop(columns=['profile'], inplace=True)
print(f"📐 Optimized Multi-Aspect Master DataFrame footprint: {len(df_master)} rows ready for split.")

📐 Optimized Multi-Aspect Master DataFrame footprint: 6946 rows ready for split.


In [ ]:
def explode_to_pairs(df_input):
    pairs = []
    for _, row in df_input.iterrows():
        for col in target_cols:
            score = int(row[col])

            # 1. THE SIGNAL FILTER
            # Skips 0 (Abstain) completely. We only retain rows where an explicit macro
            # directional sentiment exists: 1 (Positive), 2 (Negative), or 3 (Neutral).
            if score in [1, 2, 3]:

                # 2. THE TARGET RE-MAPPING
                # Converts your dataset schema labels to match standard Hugging Face
                # index logic, which expects 0-indexed integers starting from 0.
                # 1 (Pos) -> 0,  2 (Neg) -> 1,  3 (Neu) -> 2
                hf_label = score - 1

                # 3. THE TEXT SANITIZATION PATCH
                # Replaces underscores with space characters so the model processes
                # words naturally ("Real_Economic_Activity" -> "Real Economic Activity").
                clean_aspect = re.sub(r'_+', ' ', col).strip()

                # 4. THE EXPLOIDED MATRIX STRUCTURING
                # Appends an independent sequence pair directly into your list pool.
                pairs.append({
                    "text": str(row["text"]),
                    "aspect": clean_aspect,
                    "label": hf_label
                })
    return pd.DataFrame(pairs)

In [ ]:
!pip install scikit-multilearn -q
from skmultilearn.model_selection import iterative_train_test_split
import numpy as np

# Convert master features and arrays
X = df_master[['text']].values
y = df_master[target_cols].values.astype(int)

# --- SPLIT 1: Isolate Holdout Test Data (20% of global data) ---
X_temp, y_temp, X_test_raw, y_test_raw = iterative_train_test_split(X, y, test_size=0.20)

# --- SPLIT 2: Separate Remaining Data into Train & Val (80/20 of the remaining pool) ---
X_train_raw, y_train_raw, X_val_raw, y_val_raw = iterative_train_test_split(X_temp, y_temp, test_size=0.20)

# Convert arrays back into separate pristine DataFrames
df_train_raw = pd.DataFrame(X_train_raw, columns=['text'])
df_train_raw[target_cols] = y_train_raw

df_val_raw = pd.DataFrame(X_val_raw, columns=['text'])
df_val_raw[target_cols] = y_val_raw

df_test_raw = pd.DataFrame(X_test_raw, columns=['text'])
df_test_raw[target_cols] = y_test_raw

print(f"🧱 Train Base Sentences : {len(df_train_raw)}")
print(f"🧪 Val Base Sentences   : {len(df_val_raw)}")
print(f"🔒 Test Base Sentences  : {len(df_test_raw)} (LOCK AWAY UNTIL END)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 4.6 MB/s eta 0:00:00
🧱 Train Base Sentences : 4444
🧪 Val Base Sentences   : 1112
🔒 Test Base Sentences  : 1390 (LOCK AWAY UNTIL END)


In [ ]:
import re
df_train_exploded = explode_to_pairs(df_train_raw)
df_val_exploded = explode_to_pairs(df_val_raw)
df_test_exploded = explode_to_pairs(df_test_raw)

print(f"💥 Final Sequence Pairs -> Train: {len(df_train_exploded)} | Val: {len(df_val_exploded)} | Test: {len(df_test_exploded)}")

💥 Final Sequence Pairs -> Train: 7016 | Val: 1754 | Test: 2192


In [ ]:
train_df_sentiment = df_train_exploded
val_df_sentiment = df_val_exploded

In [ ]:
import os
import pandas as pd
from google.colab import drive

print("==================== 📁 MOUNTING GOOGLE DRIVE ====================")

# 1. Safely mount Google Drive to your workspace runtime instance
try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Google Drive integration verified successfully.")
except Exception as e:
    print(f"❌ Failed to interface with Google Drive: {e}")
    print("💡 Make sure you are running this inside a Google Colab notebook.")

# 2. Establish a dedicated project folder inside your Drive
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/ABSA_Data/"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)

# Define your target storage paths pointing directly to Drive
EXPORT_PATHS = {
    "Train Set": os.path.join(DRIVE_PROJECT_DIR, "finbert_absa_exploded_train.csv"),
    "Validation Set": os.path.join(DRIVE_PROJECT_DIR, "finbert_absa_exploded_val.csv"),
    "Test Set": os.path.join(DRIVE_PROJECT_DIR, "finbert_absa_exploded_test.csv")
}

print("\n==================== 💾 EXPORTING TO GDRIVE ====================")

export_map = {
    "Train Set": df_train_exploded,
    "Validation Set": df_val_exploded,
    "Test Set": df_test_exploded
}

# 3. Write dataframes straight to the persistent cloud storage directory
for name, df_target in export_map.items():
    file_path = EXPORT_PATHS[name]

    try:
        # Save sequence pair frames cleanly with UTF-8 encoding
        df_target.to_csv(file_path, index=False, encoding='utf-8')

        file_size_kb = os.path.getsize(file_path) / 1024
        print(f"💾 Saved {name:<14} -> MyDrive/ABSA_Data/{os.path.basename(file_path):<30} | {len(df_target):>5} rows | {file_size_kb:.1f} KB")

    except Exception as e:
        print(f"❌ Failed to export data channel [{name}]: {str(e)}")

print("—" * 85)
print(f"🏁 PIPELINE ASSETS SECURED UNTIL TESTING! Check your Google Drive for the 'ABSA_Data' folder.")
print("==========================================================================================")

==================== 📁 MOUNTING GOOGLE DRIVE ====================
Mounted at /content/drive
✅ Google Drive integration verified successfully.

==================== 💾 EXPORTING TO GDRIVE ====================
💾 Saved Train Set      -> MyDrive/ABSA_Data/finbert_absa_exploded_train.csv |  7016 rows | 1275.4 KB
💾 Saved Validation Set -> MyDrive/ABSA_Data/finbert_absa_exploded_val.csv  |  1754 rows | 341.9 KB
💾 Saved Test Set       -> MyDrive/ABSA_Data/finbert_absa_exploded_test.csv |  2192 rows | 429.9 KB
—————————————————————————————————————————————————————————————————————————————————————
🏁 PIPELINE ASSETS SECURED UNTIL TESTING! Check your Google Drive for the 'ABSA_Data' folder.


In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "ProsusAI/finbert"
print(f"📥 Loading foundational {MODEL_NAME} tokenizer asset...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class AspectSentimentDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # This formatting strategy forces: [CLS] text [SEP] aspect [SEP]
        inputs = self.tokenizer(
            text=str(row['text']),
            text_pair=str(row['aspect']),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        item = {key: val.squeeze(0) for key, val in inputs.items()}
        item['labels'] = torch.tensor(int(row['label']), dtype=torch.long)
        return item

# Construct live PyTorch datasets
train_dataset = AspectSentimentDataset(train_df_sentiment, tokenizer)
val_dataset = AspectSentimentDataset(val_df_sentiment, tokenizer)
print("📌 PyTorch dataset transformation maps fully compiled and cached in memory.")

📥 Loading foundational ProsusAI/finbert tokenizer asset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

📌 PyTorch dataset transformation maps fully compiled and cached in memory.


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    """
    Computes system operational performance over standard 3-class array heads
    Mapping: 0 -> Positive, 1 -> Negative, 2 -> Neutral
    """
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=1)

    # Calculate Macro F1 to evaluate balanced focus over all 3 sentiments
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "macro_f1": macro_f1
    }

print("🎯 Evaluation metric computation engines safely attached.")

🎯 Evaluation metric computation engines safely attached.


In [ ]:
import os
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Route your checkpoint directory cleanly inside your checked-out local repo path
REPO_CHECKPOINT_DIR = "/content/economic-news-sentiment/finbert_checkpoints"
os.makedirs(REPO_CHECKPOINT_DIR, exist_ok=True)

print("🔄 Loading native FinBERT baseline weights...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True # Safely swaps out the default classification layer mapping
)

training_args = TrainingArguments(
    output_dir=REPO_CHECKPOINT_DIR,    # Output updates drop straight into local repo path
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,                # Keeps folder neat by discarding legacy checkpoints instantly
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=15,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Automate Checkpoint Recovery Lookups within the Repo File System
checkpoint_flag = False
if os.path.exists(REPO_CHECKPOINT_DIR) and len(os.listdir(REPO_CHECKPOINT_DIR)) > 0:
    for sub in os.listdir(REPO_CHECKPOINT_DIR):
        if "checkpoint" in sub:
            checkpoint_flag = True

print("\n🏁 Setup Complete!")
if checkpoint_flag:
    print("➡️ Existing checkpoint found inside Git repo structure. Training states will resume safely!")
else:
    print("➡️ No local checkpoint found. Preparing a completely clean training pass.")

🔄 Loading native FinBERT baseline weights...


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


🏁 Setup Complete!
➡️ No local checkpoint found. Preparing a completely clean training pass.


In [ ]:
print("🚀 Launching FinBERT Aspect Sentiment fine-tuning loops...")

# Start training, allowing seamless crash recovery execution if flags were raised
train_metrics = trainer.train(resume_from_checkpoint=checkpoint_flag)

print("\n🎉 Fine-tuning sequence executed successfully!")
print(train_metrics)

🚀 Launching FinBERT Aspect Sentiment fine-tuning loops...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.726782,0.655114,0.725770,0.628477
2,0.573088,0.630557,0.756556,0.686960
3,0.281091,0.730750,0.751425,0.692492
4,0.151837,0.877199,0.753706,0.695449
5,0.125421,0.950865,0.762828,0.706539


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


🎉 Fine-tuning sequence executed successfully!
TrainOutput(global_step=2195, training_loss=0.38762888115467947, metrics={'train_runtime': 882.6336, 'train_samples_per_second': 39.745, 'train_steps_per_second': 2.487, 'total_flos': 2307504673474560.0, 'train_loss': 0.38762888115467947, 'epoch': 5.0})


In [ ]:
FINAL_DRIVE_OUT = "/content/drive/MyDrive/economic_news_project/final1_finbert_aspect_sentiment"
os.makedirs(FINAL_DRIVE_OUT, exist_ok=True)

print(f"💾 Exporting fully optimized best weights tracking state directly to Drive container...")

# Save clean tracking layers only—excluding massive local optimizer artifacts from Drive!
trainer.model.save_pretrained(FINAL_DRIVE_OUT)
tokenizer.save_pretrained(FINAL_DRIVE_OUT)

print("="*65)
print(f"✅ PIPELINE CONCLUDED!")
print(f"Your optimized aspect sentiment model is permanently stored in cloud space at:")
print(f"📂 {FINAL_DRIVE_OUT}")
print("="*65)

💾 Exporting fully optimized best weights tracking state directly to Drive container...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ PIPELINE CONCLUDED!
Your optimized aspect sentiment model is permanently stored in cloud space at:
📂 /content/drive/MyDrive/economic_news_project/final1_finbert_aspect_sentiment


In [ ]:
import os
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Configured to point directly to your pristine Google Drive export directory
MODEL_DRIVE_PATH = "/content/drive/MyDrive/economic_news_project/final1_finbert_aspect_sentiment"

DOMAIN_LABELS = [
    "Monetary Financial",
    "Inflation Prices",
    "Real Economic_Activity",
    "Labor Consumption",
    "Fiscal Government",
    "External Sector"
]

SENTIMENT_MAP = {0: "🟢 Positive", 1: "🔴 Negative", 2: "🟡 Neutral"}

print("🔄 Initializing interactive test pipeline. Loading weights from Google Drive...")
if not os.path.exists(MODEL_DRIVE_PATH):
    raise FileNotFoundError(f"❌ Could not find fine-tuned weights at {MODEL_DRIVE_PATH}. Did you run Cell 7?")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DRIVE_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DRIVE_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()
print(f"✅ Live Model Engine deployed successfully on: [{device.upper()}]")
print("="*70)

# 2. Interactive Loop Execution Entry
while True:
    print("\n" + "-"*50)
    user_sentence = input("✍️ Enter an economic news sentence (or type 'exit'): ").strip()
    if user_sentence.lower() == 'exit':
        print("👋 Exiting sandbox loop. Happy coding!")
        break
    if not user_sentence:
        continue

    print("\n📋 Select Target Macro Aspect Group:")
    for i, domain in enumerate(DOMAIN_LABELS):
        print(f"  [{i}] {domain}")

    try:
        selection = input("\n🔢 Enter Aspect Number (0-5): ").strip()
        aspect_idx = int(selection)
        if aspect_idx < 0 or aspect_idx >= len(DOMAIN_LABELS):
            print("❌ Invalid number choice! Resetting entry frame...")
            continue
    except ValueError:
        print("❌ Please enter a valid numerical index number!")
        continue

    chosen_aspect = DOMAIN_LABELS[aspect_idx]

    # 3. Apply your native Token-Appending Syntax Strategy: [CLS] sentence [SEP] aspect [SEP]
    inputs = tokenizer(
        text=user_sentence,
        text_pair=chosen_aspect,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    # 4. Extract Logit Vectors and Map via Softmax Distribution Scaling
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    winning_idx = np.argmax(probabilities)
    winning_label = SENTIMENT_MAP[winning_idx]
    winning_conf = probabilities[winning_idx]

    # 5. Output Multi-Class Confidence Distribution Graph
    print("\n📊 ==================== MODEL ANALYSIS ====================")
    print(f"📝 Text  : \"{user_sentence}\"")
    print(f"🎯 Aspect: {chosen_aspect}")
    print(f"🏆 Result: {winning_label} ({winning_conf*100:.2f}% Confidence)\n")
    print("📈 Full Probability Breakdown Matrix:")

    for idx, name in SENTIMENT_MAP.items():
        prob = probabilities[idx]
        bar_length = int(prob * 30)
        visual_bar = "█" * bar_length + "░" * (30 - bar_length)
        print(f"  {name:<11} | {visual_bar} | {prob*100:6.2f}%")
    print("============================================================")

🔄 Initializing interactive test pipeline. Loading weights from Google Drive...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Live Model Engine deployed successfully on: [CUDA]

--------------------------------------------------
✍️ Enter an economic news sentence (or type 'exit'): The world's biggest problem, unemployment, now no longer exists

📋 Select Target Macro Aspect Group:
  [0] Monetary Financial
  [1] Inflation Prices
  [2] Real Economic_Activity
  [3] Labor Consumption
  [4] Fiscal Government
  [5] External Sector

🔢 Enter Aspect Number (0-5): 3

📊 ==================== MODEL ANALYSIS ====================
📝 Text  : "The world's biggest problem, unemployment, now no longer exists"
🎯 Aspect: Labor Consumption
🏆 Result: 🔴 Negative (99.52% Confidence)

📈 Full Probability Breakdown Matrix:
  🟢 Positive  | ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ |   0.27%
  🔴 Negative  | █████████████████████████████░ |  99.52%
  🟡 Neutral   | ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ |   0.20%

--------------------------------------------------
✍️ Enter an economic news sentence (or type 'exit'): Inflation rate increased and grocery prices

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Initialize Baseline FinBERT Assets
BASE_MODEL = "ProsusAI/finbert"
print(f"📥 Fetching baseline {BASE_MODEL} weights from Hugging Face...")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL)
model.eval() # Set model to evaluation mode

# Mapping label IDs to match FinBERT's native configuration array
# Native FinBERT maps: 0 -> Positive, 1 -> Negative, 2 -> Neutral
LABEL_MAPPING = {0: "🟢 Positive", 1: "🔴 Negative", 2: "🟡 Neutral"}

ASPECTS = [
    "Monetary Financial",
    "Inflation Prices",
    "Real Economic Activity",
    "Labor Consumption",
    "Fiscal Government",
    "External Sector"
]

print("✅ Baseline FinBERT loaded successfully!")
print("--------------------------------------------------")

# 2. Operational Live Interactive Loop
while True:
    user_text = input("✍️ Enter an economic news sentence (or type 'exit'): ").strip()
    if user_text.lower() == 'exit':
        print("👋 Exiting sandbox loop. Happy testing!")
        break
    if not user_text:
        continue

    print("\n📋 Select Target Macro Aspect Group:")
    for idx, aspect in enumerate(ASPECTS):
        print(f"  [{idx}] {aspect}")

    try:
        aspect_num = input("\n🔢 Enter Aspect Number (0-5): ").strip()
        if aspect_num.lower() == 'exit': break
        aspect_idx = int(aspect_num)
        if aspect_idx < 0 or aspect_idx >= len(ASPECTS):
            print("❌ Invalid selection. Restarting row pass.")
            continue
    except ValueError:
        print("❌ Please enter a valid index number.")
        continue

    selected_aspect = ASPECTS[aspect_idx]

    # 3. Format exactly like your fine-tuning pipeline: [CLS] text [SEP] aspect [SEP]
    with torch.no_grad():
        inputs = tokenizer(
            text=user_text,
            text_pair=selected_aspect,
            max_length=128,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # Forward pass through baseline network layers
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = F.softmax(logits, dim=-1).squeeze(0).tolist()

    # Get highest probability index choice
    prediction_idx = probabilities.index(max(probabilities))
    winning_label = LABEL_MAPPING[prediction_idx]

    # 4. Render Analytical Matrix Dashboard
    print("\n📊 ==================== BASELINE ANALYSIS ====================")
    print(f"📝 Text  : \"{user_text}\"")
    print(f"🎯 Aspect: {selected_aspect}")
    print(f"🏆 Result: {winning_label} ({probabilities[prediction_idx]*100:.2f}% Confidence)")
    print("\n📈 Full Probability Breakdown Matrix:")

    for idx, label_name in LABEL_MAPPING.items():
        bar_count = int(probabilities[idx] * 30)
        bar_visual = "█" * bar_count + "░" * (30 - bar_count)
        print(f"  {label_name:<11} | {bar_visual} |  {probabilities[idx]*100:5.2f}%")
    print("============================================================\n")

📥 Fetching baseline ProsusAI/finbert weights from Hugging Face...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Baseline FinBERT loaded successfully!
--------------------------------------------------
✍️ Enter an economic news sentence (or type 'exit'): Unemployment at an all time low

📋 Select Target Macro Aspect Group:
  [0] Monetary Financial
  [1] Inflation Prices
  [2] Real Economic Activity
  [3] Labor Consumption
  [4] Fiscal Government
  [5] External Sector

🔢 Enter Aspect Number (0-5): 3

📊 ==================== BASELINE ANALYSIS ====================
📝 Text  : "Unemployment at an all time low"
🎯 Aspect: Labor Consumption
🏆 Result: 🔴 Negative (59.29% Confidence)

📈 Full Probability Breakdown Matrix:
  🟢 Positive  | ██░░░░░░░░░░░░░░░░░░░░░░░░░░░░ |   8.08%
  🔴 Negative  | █████████████████░░░░░░░░░░░░░ |  59.29%
  🟡 Neutral   | █████████░░░░░░░░░░░░░░░░░░░░░ |  32.63%

✍️ Enter an economic news sentence (or type 'exit'): Inflation rate increased and grocery prices increased sharply, however the government rolled out stimulus checks to accommodate households.

📋 Select Target Macro Aspect

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Initialize Baseline FinBERT Assets for Global Tracking
BASE_MODEL = "ProsusAI/finbert"
print(f"📥 Fetching baseline {BASE_MODEL} weights from Hugging Face...")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL)
model.eval()

# Native FinBERT configuration alignment mapping
LABEL_MAPPING = {0: "🟢 Positive", 1: "🔴 Negative", 2: "🟡 Neutral"}

print("✅ Global FinBERT loaded successfully!")
print("--------------------------------------------------")

# 2. Operational Live Interactive Loop
while True:
    user_text = input("✍️ Enter an economic news sentence (or type 'exit'): ").strip()
    if user_text.lower() == 'exit':
        print("👋 Exiting sandbox loop. Happy testing!")
        break
    if not user_text:
        continue

    # 3. Format strictly as a Single Sequence Pass (No Text Pair / Aspect)
    with torch.no_grad():
        inputs = tokenizer(
            text=user_text,
            max_length=128,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = F.softmax(logits, dim=-1).squeeze(0).tolist()

    # Get highest probability index choice
    prediction_idx = probabilities.index(max(probabilities))
    winning_label = LABEL_MAPPING[prediction_idx]

    # 4. Render Global Analytical Matrix Dashboard
    print("\n📊 ================= GLOBAL SENTIMENT ANALYSIS =================")
    print(f"📝 Text  : \"{user_text}\"")
    print(f"🏆 Result: {winning_label} ({probabilities[prediction_idx]*100:.2f}% Confidence)")
    print("\n📈 Full Probability Breakdown Matrix:")

    for idx, label_name in LABEL_MAPPING.items():
        bar_count = int(probabilities[idx] * 30)
        bar_visual = "█" * bar_count + "░" * (30 - bar_count)
        print(f"  {label_name:<11} | {bar_visual} |  {probabilities[idx]*100:5.2f}%")
    print("============================================================\n")

📥 Fetching baseline ProsusAI/finbert weights from Hugging Face...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Global FinBERT loaded successfully!
--------------------------------------------------
✍️ Enter an economic news sentence (or type 'exit'): Unemployment at an all time low

📊 ================= GLOBAL SENTIMENT ANALYSIS =================
📝 Text  : "Unemployment at an all time low"
🏆 Result: 🔴 Negative (64.29% Confidence)

📈 Full Probability Breakdown Matrix:
  🟢 Positive  | ████░░░░░░░░░░░░░░░░░░░░░░░░░░ |  15.36%
  🔴 Negative  | ███████████████████░░░░░░░░░░░ |  64.29%
  🟡 Neutral   | ██████░░░░░░░░░░░░░░░░░░░░░░░░ |  20.35%

✍️ Enter an economic news sentence (or type 'exit'): The world's biggest problem, unemployment, now no longer exists

📊 ================= GLOBAL SENTIMENT ANALYSIS =================
📝 Text  : "The world's biggest problem, unemployment, now no longer exists"
🏆 Result: 🔴 Negative (69.95% Confidence)

📈 Full Probability Breakdown Matrix:
  🟢 Positive  | █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ |   3.67%
  🔴 Negative  | ████████████████████░░░░░░░░░░ |  69.95%
  🟡 Neutral   |

In [9]:
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import classification_report, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# =====================================================================
# 📁 STEP 1: PATH REGISTERS & DATA ENGINE LOADER
# =====================================================================

VAL_DATA_PATH = "/content/drive/MyDrive/ABSA_Data/finbert_absa_exploded_val.csv"
MODEL_DRIVE_PATH = "/content/drive/MyDrive/economic_news_project/final1_finbert_aspect_sentiment"
BASE_MODEL_NAME = "ProsusAI/finbert"

SENTIMENT_MAP = {0: "🟢 Positive", 1: "🔴 Negative", 2: "🟡 Neutral"}

# Ultra-robust dictionary map catching strings, integers, and floats
INVERSE_SENTIMENT_MAP = {
    "positive": 0, "negative": 1, "neutral": 2,
    "0": 0, "1": 1, "2": 2,
    0: 0, 1: 1, 2: 2,
    0.0: 0, 1.0: 1, 2.0: 2
}

print("📥 Loading validation matrix from Google Drive...")
try:
    val_df = pd.read_csv(VAL_DATA_PATH)
    print(f"✅ Successfully loaded validation dataset: {len(val_df)} rows discovered.")
except Exception as e:
    print(f"❌ Failed to locate the verification file at {VAL_DATA_PATH}. Error: {e}")
    raise e

# Ensure required tracking columns exist
required_cols = ["text", "aspect", "label"]
for col in required_cols:
    if col not in val_df.columns:
        raise ValueError(f"Missing critical data layout tracking vector: {col}")

# 1. Clean up strings and safely map values
# Using a lambda to safely strip strings but pass numbers straight through
def safe_map(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, str):
        val = val.strip().lower()
    return INVERSE_SENTIMENT_MAP.get(val, np.nan)

val_df["mapped_label"] = val_df["label"].apply(safe_map)

# 2. Check if any NaNs exist and drop them so scikit-learn doesn't crash
nan_count = val_df["mapped_label"].isna().sum()
if nan_count > 0:
    print(f"⚠️ Warning: Found {nan_count} rows with invalid/NaN labels. Dropping them from evaluation.")
    # Keep only rows that mapped perfectly
    val_df = val_df.dropna(subset=["mapped_label"])
    print(f"📊 Cleaned dataset size: {len(val_df)} rows remaining.")

# 3. Finalize true targets as clean integers
y_true = val_df["mapped_label"].astype(int).tolist()
# =====================================================================
# 📦 STEP 2: LOAD PIPELINE NETWORKS (BASE VS. FINE-TUNED)
# =====================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚙️ Initializing computing engines on: {device.upper()}")

print("\n🤖 Loading Fine-Tuned ABSA Model...")
tuned_tokenizer = AutoTokenizer.from_pretrained(MODEL_DRIVE_PATH)
tuned_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DRIVE_PATH).to(device).eval()

print("🤖 Loading Baseline FinBERT Model...")
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
base_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_NAME).to(device).eval()

# =====================================================================
# 🧠 STEP 3: BATCHED EVALUATION EXECUTION PASS
# =====================================================================
def get_predictions(tokenizer, model, texts, aspects=None, is_absa=False):
    preds = []
    # Using small sequential chunks to safeguard against memory overflow crashes
    batch_size = 32

    for i in tqdm(range(0, len(texts), batch_size), desc="Running Inference"):
        batch_texts = texts[i:i + batch_size]

        if is_absa and aspects is not None:
            batch_aspects = aspects[i:i + batch_size]
            # ABSA Model expects: text="text string", text_pair="aspect string"
            inputs = tokenizer(
                text=batch_texts,
                text_pair=batch_aspects,
                padding="max_length",
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(device)
        else:
            # Baseline FinBERT model only processes the global raw text block
            inputs = tokenizer(
                text=batch_texts,
                padding="max_length",
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            # Find the index of the highest logit per row
            batch_preds = torch.argmax(outputs.logits, dim=-1).cpu().tolist()
            preds.extend(batch_preds)

    return preds

text_list = val_df["text"].astype(str).tolist()
aspects_list = val_df["aspect"].astype(str).tolist()

print("\n🚀 Commencing Evaluation Pass 1/2: Fine-Tuned Aspect-Based Model...")
y_pred_tuned = get_predictions(tuned_tokenizer, tuned_model, text_list, aspects=aspects_list, is_absa=True)

print("\n🚀 Commencing Evaluation Pass 2/2: Baseline Global FinBERT Model...")
y_pred_base = get_predictions(base_tokenizer, base_model, text_list, is_absa=False)

# =====================================================================
# 📊 STEP 4: METRIC LOG COMPILATION DISPLAY
# =====================================================================
target_names = ["Positive", "Negative", "Neutral"]

print("\n" + "="*30 + " 📊 BASELINE FINBERT METRICS " + "="*30)
print(f"Global Macro F1 Score: {f1_score(y_true, y_pred_base, average='macro')*100:.2f}%")
print(f"Global Micro F1 Score: {f1_score(y_true, y_pred_base, average='micro')*100:.2f}%")
print("-" * 89)
print(classification_report(y_true, y_pred_base, target_names=target_names, digits=4))

print("\n" + "="*30 + " 🚀 FINE-TUNED ABSA MODEL METRICS " + "="*30)
print(f"Global Macro F1 Score: {f1_score(y_true, y_pred_tuned, average='macro')*100:.2f}%")
print(f"Global Micro F1 Score: {f1_score(y_true, y_pred_tuned, average='micro')*100:.2f}%")
print("-" * 89)
print(classification_report(y_true, y_pred_tuned, target_names=target_names, digits=4))

📥 Loading validation matrix from Google Drive...
✅ Successfully loaded validation dataset: 1754 rows discovered.
⚙️ Initializing computing engines on: CUDA

🤖 Loading Fine-Tuned ABSA Model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

🤖 Loading Baseline FinBERT Model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


🚀 Commencing Evaluation Pass 1/2: Fine-Tuned Aspect-Based Model...


Running Inference: 100%|██████████| 55/55 [00:11<00:00,  4.69it/s]



🚀 Commencing Evaluation Pass 2/2: Baseline Global FinBERT Model...


Running Inference: 100%|██████████| 55/55 [00:12<00:00,  4.54it/s]


============================== 📊 BASELINE FINBERT METRICS ==============================
Global Macro F1 Score: 56.07%
Global Micro F1 Score: 61.86%
-----------------------------------------------------------------------------------------
              precision    recall  f1-score   support

    Positive     0.6744    0.6644    0.6694       873
    Negative     0.7067    0.6224    0.6618       662
     Neutral     0.2990    0.4247    0.3509       219

    accuracy                         0.6186      1754
   macro avg     0.5600    0.5705    0.5607      1754
weighted avg     0.6397    0.6186    0.6268      1754


============================== 🚀 FINE-TUNED ABSA MODEL METRICS ==============================
Global Macro F1 Score: 70.65%
Global Micro F1 Score: 76.28%
-----------------------------------------------------------------------------------------
              precision    recall  f1-score   support

    Positive     0.8219    0.7663    0.7931       873
    Negative     0.8227  

In [10]:
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import classification_report, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# =====================================================================
# 📁 STEP 1: PATH REGISTERS & DATA ENGINE LOADER
# =====================================================================

VAL_DATA_PATH = "/content/drive/MyDrive/ABSA_Data/finbert_absa_exploded_test.csv"
MODEL_DRIVE_PATH = "/content/drive/MyDrive/economic_news_project/final1_finbert_aspect_sentiment"
BASE_MODEL_NAME = "ProsusAI/finbert"

SENTIMENT_MAP = {0: "🟢 Positive", 1: "🔴 Negative", 2: "🟡 Neutral"}

# Ultra-robust dictionary map catching strings, integers, and floats
INVERSE_SENTIMENT_MAP = {
    "positive": 0, "negative": 1, "neutral": 2,
    "0": 0, "1": 1, "2": 2,
    0: 0, 1: 1, 2: 2,
    0.0: 0, 1.0: 1, 2.0: 2
}

print("📥 Loading validation matrix from Google Drive...")
try:
    val_df = pd.read_csv(VAL_DATA_PATH)
    print(f"✅ Successfully loaded validation dataset: {len(val_df)} rows discovered.")
except Exception as e:
    print(f"❌ Failed to locate the verification file at {VAL_DATA_PATH}. Error: {e}")
    raise e

# Ensure required tracking columns exist
required_cols = ["text", "aspect", "label"]
for col in required_cols:
    if col not in val_df.columns:
        raise ValueError(f"Missing critical data layout tracking vector: {col}")

# 1. Clean up strings and safely map values
# Using a lambda to safely strip strings but pass numbers straight through
def safe_map(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, str):
        val = val.strip().lower()
    return INVERSE_SENTIMENT_MAP.get(val, np.nan)

val_df["mapped_label"] = val_df["label"].apply(safe_map)

# 2. Check if any NaNs exist and drop them so scikit-learn doesn't crash
nan_count = val_df["mapped_label"].isna().sum()
if nan_count > 0:
    print(f"⚠️ Warning: Found {nan_count} rows with invalid/NaN labels. Dropping them from evaluation.")
    # Keep only rows that mapped perfectly
    val_df = val_df.dropna(subset=["mapped_label"])
    print(f"📊 Cleaned dataset size: {len(val_df)} rows remaining.")

# 3. Finalize true targets as clean integers
y_true = val_df["mapped_label"].astype(int).tolist()
# =====================================================================
# 📦 STEP 2: LOAD PIPELINE NETWORKS (BASE VS. FINE-TUNED)
# =====================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚙️ Initializing computing engines on: {device.upper()}")

print("\n🤖 Loading Fine-Tuned ABSA Model...")
tuned_tokenizer = AutoTokenizer.from_pretrained(MODEL_DRIVE_PATH)
tuned_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DRIVE_PATH).to(device).eval()

print("🤖 Loading Baseline FinBERT Model...")
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
base_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_NAME).to(device).eval()

# =====================================================================
# 🧠 STEP 3: BATCHED EVALUATION EXECUTION PASS
# =====================================================================
def get_predictions(tokenizer, model, texts, aspects=None, is_absa=False):
    preds = []
    # Using small sequential chunks to safeguard against memory overflow crashes
    batch_size = 32

    for i in tqdm(range(0, len(texts), batch_size), desc="Running Inference"):
        batch_texts = texts[i:i + batch_size]

        if is_absa and aspects is not None:
            batch_aspects = aspects[i:i + batch_size]
            # ABSA Model expects: text="text string", text_pair="aspect string"
            inputs = tokenizer(
                text=batch_texts,
                text_pair=batch_aspects,
                padding="max_length",
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(device)
        else:
            # Baseline FinBERT model only processes the global raw text block
            inputs = tokenizer(
                text=batch_texts,
                padding="max_length",
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            # Find the index of the highest logit per row
            batch_preds = torch.argmax(outputs.logits, dim=-1).cpu().tolist()
            preds.extend(batch_preds)

    return preds

text_list = val_df["text"].astype(str).tolist()
aspects_list = val_df["aspect"].astype(str).tolist()

print("\n🚀 Commencing Evaluation Pass 1/2: Fine-Tuned Aspect-Based Model...")
y_pred_tuned = get_predictions(tuned_tokenizer, tuned_model, text_list, aspects=aspects_list, is_absa=True)

print("\n🚀 Commencing Evaluation Pass 2/2: Baseline Global FinBERT Model...")
y_pred_base = get_predictions(base_tokenizer, base_model, text_list, is_absa=False)

# =====================================================================
# 📊 STEP 4: METRIC LOG COMPILATION DISPLAY
# =====================================================================
target_names = ["Positive", "Negative", "Neutral"]

print("\n" + "="*30 + " 📊 BASELINE FINBERT METRICS " + "="*30)
print(f"Global Macro F1 Score: {f1_score(y_true, y_pred_base, average='macro')*100:.2f}%")
print(f"Global Micro F1 Score: {f1_score(y_true, y_pred_base, average='micro')*100:.2f}%")
print("-" * 89)
print(classification_report(y_true, y_pred_base, target_names=target_names, digits=4))

print("\n" + "="*30 + " 🚀 FINE-TUNED ABSA MODEL METRICS " + "="*30)
print(f"Global Macro F1 Score: {f1_score(y_true, y_pred_tuned, average='macro')*100:.2f}%")
print(f"Global Micro F1 Score: {f1_score(y_true, y_pred_tuned, average='micro')*100:.2f}%")
print("-" * 89)
print(classification_report(y_true, y_pred_tuned, target_names=target_names, digits=4))

📥 Loading validation matrix from Google Drive...
✅ Successfully loaded validation dataset: 2192 rows discovered.
⚙️ Initializing computing engines on: CUDA

🤖 Loading Fine-Tuned ABSA Model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

🤖 Loading Baseline FinBERT Model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


🚀 Commencing Evaluation Pass 1/2: Fine-Tuned Aspect-Based Model...


Running Inference: 100%|██████████| 69/69 [00:14<00:00,  4.64it/s]



🚀 Commencing Evaluation Pass 2/2: Baseline Global FinBERT Model...


Running Inference: 100%|██████████| 69/69 [00:14<00:00,  4.68it/s]


============================== 📊 BASELINE FINBERT METRICS ==============================
Global Macro F1 Score: 56.95%
Global Micro F1 Score: 63.14%
-----------------------------------------------------------------------------------------
              precision    recall  f1-score   support

    Positive     0.6976    0.6645    0.6807      1097
    Negative     0.7361    0.6954    0.7151       778
     Neutral     0.2767    0.3596    0.3128       317

    accuracy                         0.6314      2192
   macro avg     0.5701    0.5732    0.5695      2192
weighted avg     0.6504    0.6314    0.6397      2192


============================== 🚀 FINE-TUNED ABSA MODEL METRICS ==============================
Global Macro F1 Score: 64.22%
Global Micro F1 Score: 70.39%
-----------------------------------------------------------------------------------------
              precision    recall  f1-score   support

    Positive     0.7531    0.7256    0.7391      1097
    Negative     0.7826  